Source for TF-IDF:
https://towardsdatascience.com/measure-text-weight-using-tf-idf-in-python-plain-code-and-scikit-learn-50cb1e4375ad/

In [1]:
!wget -nc https://github.com/TurkuNLP/intro-to-nlp/raw/master/Data/imdb_train.json

File ‘imdb_train.json’ already there; not retrieving.



In [2]:
import json # JSON encoder and decoder: store python data structures (e.g. lists and dictionaries) as strings

with open("imdb_train.json", "rt", encoding="utf-8") as f:
    data = json.load(f)

print("Data type:", type(data))
print("First item type:", type(data[0]))
print("First item:", data[0])

Data type: <class 'list'>
First item type: <class 'dict'>
First item: {'class': 'pos', 'text': "With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.  Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if t

In [3]:
import tqdm
from collections import Counter
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
token_counter = Counter()

for doc in tqdm.tqdm(data[:1000]): # IMDB documents
    tokenized = tokenizer.tokenize(doc["text"])
    token_counter.update(tokenized)

print("Number of tokens in total:", token_counter.total())
print("Most common tokens:")
for item in token_counter.most_common(20):
  print(item)
print("Vocabulary size:", len(token_counter))

100%|██████████| 1000/1000 [00:00<00:00, 3175.51it/s]

Number of tokens in total: 296082
Most common tokens:
('the', 13266)
('.', 12777)
(',', 11014)
('a', 6580)
('and', 6560)
('of', 5796)
("'", 5337)
('to', 5324)
('is', 4247)
('it', 3763)
('in', 3716)
('i', 3375)
('this', 2889)
('that', 2843)
('-', 2767)
('s', 2649)
('\\', 2639)
('"', 2638)
('was', 1944)
('with', 1866)
Vocabulary size: 14858


IDF weight function:

In [5]:
import math

def calculateIDF(tokenInDocument: int, documentInt: int):
    if tokenInDocument == 0:
        return 0
    
    return float(math.log(documentInt/tokenInDocument))

Getting the IDF weights for individual tokens:

In [6]:
import re

idfDict = dict()
tokenInDocument: int = 0

for token in token_counter.keys():
    for doc in tqdm.tqdm(data[:1000], disable=True):
        if "#" in token:
            modToken = token.replace('#','')
            pattern = re.compile(r"/w*" + modToken)
            if re.search(pattern, doc["text"]) == True:
                tokenInDocument += 1
                break
            else:
                break

        elif token in doc["text"]:
            tokenInDocument += 1
            break
    
    idfDict[token] = float(calculateIDF(tokenInDocument, 1000))
    tokenInDocument = 0

print(idfDict)

{'with': 6.907755278982137, 'all': 6.907755278982137, 'this': 6.907755278982137, 'stuff': 6.907755278982137, 'going': 6.907755278982137, 'down': 6.907755278982137, 'at': 6.907755278982137, 'the': 6.907755278982137, 'moment': 6.907755278982137, 'm': 6.907755278982137, '##j': 0.0, 'i': 6.907755278982137, "'": 6.907755278982137, 've': 6.907755278982137, 'started': 6.907755278982137, 'listening': 6.907755278982137, 'to': 6.907755278982137, 'his': 6.907755278982137, 'music': 6.907755278982137, ',': 6.907755278982137, 'watching': 6.907755278982137, 'odd': 6.907755278982137, 'documentary': 6.907755278982137, 'here': 6.907755278982137, 'and': 6.907755278982137, 'there': 6.907755278982137, 'watched': 6.907755278982137, 'wi': 6.907755278982137, '##z': 0.0, 'moon': 6.907755278982137, '##walker': 0.0, 'again': 6.907755278982137, '.': 6.907755278982137, 'maybe': 6.907755278982137, 'just': 6.907755278982137, 'want': 6.907755278982137, 'get': 6.907755278982137, 'a': 6.907755278982137, 'certain': 6.90

In [7]:
def calculateTFIDF(idf, tokenInt, allTokens):
    if tokenInt == 0:
        return 0
    if idf == 0:
        return 0
    
    return float(tokenInt/allTokens*idf)

In [8]:
token_counter.clear()
alltfIDF = dict()
i = 0

for doc in tqdm.tqdm(data[:1000], disable=True):
    tfIDFList = dict()
    tokenized = tokenizer.tokenize(doc["text"])
    token_counter.update(tokenized)
    tokens = token_counter.keys()
    for token in tokens:
        try:
            tfIdf = calculateTFIDF(idfDict.get(token), token_counter.get(token), len(token_counter))
            tfIDFList[token] = tfIdf
        except:
            print(idfDict.get(token))
    
    alltfIDF[i] = tfIDFList
    i =+ 1

In [9]:
i = 0

for tokens in alltfIDF.values():
    print(f"Document {i}'s highest TF-IDF weights: {dict(sorted(tokens.items(), key=lambda item: item[1]))}")
    i =+ 1

Document 0's highest TF-IDF weights: {'##j': 0, '##z': 0, '##walker': 0, 'jackson': 0, '##tist': 0, '##ing': 0, '##sc': 0, '##i': 0, '##pathic': 0, '##ted': 0, '##o': 0, '##y': 0, '##some': 0, '##est': 0, '##m': 0, '##s': 0, 'stuff': 0.027853851931379583, 'down': 0.027853851931379583, 'moment': 0.027853851931379583, 'started': 0.027853851931379583, 'listening': 0.027853851931379583, 'watching': 0.027853851931379583, 'odd': 0.027853851931379583, 'documentary': 0.027853851931379583, 'here': 0.027853851931379583, 'there': 0.027853851931379583, 'wi': 0.027853851931379583, 'again': 0.027853851931379583, 'want': 0.027853851931379583, 'get': 0.027853851931379583, 'certain': 0.027853851931379583, 'insight': 0.027853851931379583, 'thought': 0.027853851931379583, 'eighties': 0.027853851931379583, 'make': 0.027853851931379583, 'up': 0.027853851931379583, 'my': 0.027853851931379583, 'mind': 0.027853851931379583, 'whether': 0.027853851931379583, 'innocent': 0.027853851931379583, 'biography': 0.0278